1. Mounting the Drive and loading data features

In [2]:
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/mzazicare-ml'
DATA = f'{BASE}/data'
MODELS = f'{BASE}/models'

import os
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

os.makedirs(MODELS, exist_ok=True)

df = pd.read_parquet(f'{DATA}/rba_features_v2.parquet')
print(f"Loaded: {df.shape}")
print(df.columns.tolist())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded: (1563463, 16)
['login_hour', 'login_day_of_week', 'time_deviation', 'is_known_country', 'is_known_city', 'is_foreign_login', 'location_anomaly_score', 'is_known_device', 'login_velocity', 'hours_since_last_login', 'is_new_asn', 'is_new_browser_os', 'is_new_device_type', 'Is Attack IP', 'Login Successful', 'Is Account Takeover']


2. Separating features and labels

In [11]:
features = [
    'login_hour',
    'login_day_of_week',
    'time_deviation',
    'is_known_country',
    'is_known_city',
    'is_foreign_login',
    'location_anomaly_score',
    'is_known_device',
    'login_velocity',
    'hours_since_last_login'
]

labels = ['Is Attack IP', 'Is Account Takeover', 'Login Successful']

X = df[features].copy()
y = df[labels].copy()

print(f"Feature matrix shape: {X.shape}")
print(f"Labels shape: {y.shape}")
print(f"\nAttack IPs: {y['Is Attack IP'].sum():,}")
print(f"Account takeovers: {y['Is Account Takeover'].sum():,}")
print(f"Successful logins: {y['Login Successful'].sum():,}")
print(f"\nAnomaly rate (attack IP): {y['Is Attack IP'].mean()*100:.4f}%")

Feature matrix shape: (1563463, 10)
Labels shape: (1563463, 3)

Attack IPs: 155,004
Account takeovers: 7
Successful logins: 626,734

Anomaly rate (attack IP): 9.9141%


6b. Feature comparison before retraining the model

In [12]:
import matplotlib.pyplot as plt

# Checking feature separability; whether features actually differ between attacks and normal
new_features_check = features  # including hours_since_last_login
comparison_df = df[new_features_check + ['Is Attack IP']].dropna()

comparison = comparison_df.groupby('Is Attack IP')[new_features_check].mean().T
comparison.columns = ['Normal', 'Attack']
comparison['difference'] = abs(comparison['Attack'] - comparison['Normal'])
comparison = comparison.sort_values('difference', ascending=False)
print(comparison.round(4))

                          Normal   Attack  difference
hours_since_last_login  201.4616  59.7452    141.7165
login_velocity           44.6339  49.7649      5.1310
login_hour               12.3756  10.9012      1.4744
location_anomaly_score    0.6285   0.2829      0.3456
is_foreign_login          0.3209   0.1366      0.1843
is_known_country          0.6791   0.8634      0.1843
is_known_city             0.6924   0.8537      0.1613
login_day_of_week         2.8790   2.9863      0.1073
is_known_device           0.4325   0.4804      0.0479
time_deviation            2.8086   2.8094      0.0008


3. Scaling features

In [13]:
# Log-transforming the skewed feature to reduce outlier influence
# Guarding against double-application if this cell is re-run
if X['hours_since_last_login'].max() > 20:  # raw hours can be large; log values won't be
    X['hours_since_last_login'] = np.log1p(X['hours_since_last_login'])
    print("Log-transform applied")
else:
    print("Log-transform already applied — skipping")

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=features)

print("Features scaled successfully")
print(X_scaled.describe().round(4))

Log-transform applied
Features scaled successfully
         login_hour  login_day_of_week  time_deviation  is_known_country  \
count  1.563463e+06       1.563463e+06    1.563463e+06      1.563463e+06   
mean  -0.000000e+00      -0.000000e+00    0.000000e+00     -0.000000e+00   
std    1.000000e+00       1.000000e+00    1.000000e+00      1.000000e+00   
min   -2.153400e+00      -1.457400e+00   -8.499000e-01     -1.518000e+00   
25%   -7.447000e-01      -9.531000e-01   -8.499000e-01     -1.518000e+00   
50%   -4.040000e-02       5.570000e-02   -3.960000e-01      6.588000e-01   
75%    8.400000e-01       1.064400e+00    6.016000e-01      6.588000e-01   
max    1.896500e+00       1.568700e+00    5.025800e+00      6.588000e-01   

       is_known_city  is_foreign_login  location_anomaly_score  \
count   1.563463e+06      1.563463e+06            1.563463e+06   
mean   -0.000000e+00      0.000000e+00            0.000000e+00   
std     1.000000e+00      1.000000e+00            1.000000e+00   


4. Train/test split

In [14]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=0.2,
    random_state=42,
    stratify=y['Is Attack IP']  # ensure attack IPs are represented in both sets
)

print(f"Training set: {X_train.shape}")
print(f"Test set:     {X_test.shape}")
print(f"\nAttack IPs in training: {y_train['Is Attack IP'].sum():,}")
print(f"Attack IPs in test:     {y_test['Is Attack IP'].sum():,}")

Training set: (1250770, 10)
Test set:     (312693, 10)

Attack IPs in training: 124,003
Attack IPs in test:     31,001


5. Determining contamination parameters

In [15]:
# Contamination = expected proportion of anomalies in training data
# This tells Isolation Forest what fraction of points to treat as outliers
# Using a contamination value of 0.05 because
# Isolation Forest works best between 0.01 and 0.05

contamination = 0.05
print(f"Contamination parameter set to:      {contamination}")
print("Rationale: prioritising precision to reduce false alarms")

Contamination parameter set to:      0.05
Rationale: prioritising precision to reduce false alarms


6. Training the Isolation Forest

In [16]:
from sklearn.ensemble import IsolationForest
import time

print("Training Isolation Forest...")
start = time.time()

model = IsolationForest(
    n_estimators=200,        # increasing number of trees for stability
    contamination=contamination,
    max_samples=256,
    max_features=0.8,
    random_state=42,
    n_jobs=-1,               # use all CPU cores
    verbose=1
)

model.fit(X_train)

elapsed = time.time() - start
print(f"\nTraining complete in {elapsed:.1f} seconds")
print(f"Number of estimators: {model.n_estimators}")
print(f"Contamination: {model.contamination}")

Training Isolation Forest...


[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done   2 out of   2 | elapsed:    4.0s finished
[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    4.5s



Training complete in 20.5 seconds
Number of estimators: 200
Contamination: 0.05


[Parallel(n_jobs=1)]: Done 199 tasks      | elapsed:   16.3s
[Parallel(n_jobs=1)]: Done 200 out of 200 | elapsed:   16.4s finished


7. Generating predictions and scores

In [17]:
# Isolation Forest returns:
# predict():         1 = normal, -1 = anomaly
# decision_function(): negative score = more anomalous, positive = more normal
# score_samples():   raw anomaly score

print("Generating predictions on test set...")

y_pred = model.predict(X_test)           # 1 = normal, -1 = anomaly
decision_scores = model.decision_function(X_test)  # higher = more normal

# Convert to binary: 1 = anomaly (attack), 0 = normal
y_pred_binary = (y_pred == -1).astype(int)

# Convert decision scores to 0-100 risk score
# More negative decision score = higher risk
risk_scores = 1 - (decision_scores - decision_scores.min()) / (
    decision_scores.max() - decision_scores.min()
)
risk_scores_100 = (risk_scores * 100).round(2)

print(f"Predictions generated: {len(y_pred_binary):,}")
print(f"\nPredicted anomalies: {y_pred_binary.sum():,} ({y_pred_binary.mean()*100:.2f}%)")
print(f"\nRisk score distribution:")
print(pd.Series(risk_scores_100).describe().round(2))

Generating predictions on test set...


[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    0.8s
[Parallel(n_jobs=1)]: Done 199 tasks      | elapsed:    3.3s
[Parallel(n_jobs=1)]: Done 200 out of 200 | elapsed:    3.3s finished
[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    0.8s


Predictions generated: 312,693

Predicted anomalies: 15,653 (5.01%)

Risk score distribution:
count    312693.00
mean         33.35
std          23.43
min           0.00
25%           8.57
50%          37.56
75%          52.02
max         100.00
dtype: float64


[Parallel(n_jobs=1)]: Done 199 tasks      | elapsed:    3.8s
[Parallel(n_jobs=1)]: Done 200 out of 200 | elapsed:    3.8s finished


8. Evaluating the model

In [18]:
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    roc_auc_score
)

y_true = y_test['Is Attack IP'].values

print("=" * 55)
print("MODEL EVALUATION REPORT")
print("=" * 55)

precision = precision_score(y_true, y_pred_binary, zero_division=0)
recall = recall_score(y_true, y_pred_binary, zero_division=0)
f1 = f1_score(y_true, y_pred_binary, zero_division=0)
false_positive_rate = (
    ((y_pred_binary == 1) & (y_true == 0)).sum() /
    (y_true == 0).sum()
)

print(f"\nPrecision:          {precision:.4f}  (target: > 0.80)")
print(f"Recall:             {recall:.4f}  (target: > 0.85)")
print(f"F1 Score:           {f1:.4f}  (target: > 0.82)")
print(f"False Positive Rate:{false_positive_rate:.4f}  (target: < 0.05)")

# Check targets met
print("\nTargets met:")
print(f"  Precision > 0.80:  {'PASS' if precision > 0.80 else 'FAIL'}")
print(f"  Recall > 0.85:     {'PASS' if recall > 0.85 else 'FAIL'}")
print(f"  F1 > 0.82:         {'PASS' if f1 > 0.82 else 'FAIL'}")
print(f"  FPR < 0.05:        {'PASS' if false_positive_rate < 0.05 else 'FAIL'}")

print("\nConfusion Matrix:")
cm = confusion_matrix(y_true, y_pred_binary)
print(f"  True Negatives  (normal, predicted normal):  {cm[0][0]:,}")
print(f"  False Positives (normal, predicted attack):  {cm[0][1]:,}")
print(f"  False Negatives (attack, predicted normal):  {cm[1][0]:,}")
print(f"  True Positives  (attack, predicted attack):  {cm[1][1]:,}")

print("\nFull Classification Report:")
print(classification_report(y_true, y_pred_binary,
      target_names=['Normal', 'Attack']))

MODEL EVALUATION REPORT

Precision:          0.0682  (target: > 0.80)
Recall:             0.0345  (target: > 0.85)
F1 Score:           0.0458  (target: > 0.82)
False Positive Rate:0.0518  (target: < 0.05)

Targets met:
  Precision > 0.80:  FAIL
  Recall > 0.85:     FAIL
  F1 > 0.82:         FAIL
  FPR < 0.05:        FAIL

Confusion Matrix:
  True Negatives  (normal, predicted normal):  267,107
  False Positives (normal, predicted attack):  14,585
  False Negatives (attack, predicted normal):  29,933
  True Positives  (attack, predicted attack):  1,068

Full Classification Report:
              precision    recall  f1-score   support

      Normal       0.90      0.95      0.92    281692
      Attack       0.07      0.03      0.05     31001

    accuracy                           0.86    312693
   macro avg       0.48      0.49      0.48    312693
weighted avg       0.82      0.86      0.84    312693

